In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
import shap
import matplotlib.pyplot as plt
import os

# Garantir que a pasta de figuras existe
os.makedirs('figuras', exist_ok=True)

# Configurações globais de fonte (replicando analise_exploratoria.Rmd)
plt.rcParams.update({'font.size': 14})
plt.rc('axes', labelsize=22)
plt.rc('xtick', labelsize=14)
plt.rc('ytick', labelsize=14)

datasets = {
    'Produto': 'data/product.csv',
    'Pais': 'data/country.csv',
    'Cliente': 'data/customer.csv'
}

for serie_name, path in datasets.items():
    print(f"\n========================================")
    print(f"Processando série: {serie_name}")
    print(f"========================================")
    
    df = pd.read_csv(path, parse_dates=['Date']).set_index('Date')
    
    target = 'Revenue'
    y = df[target]
    
    # Removendo a variável alvo e as temporais puras
    cols_to_drop = [target, 'Year', 'Quarter', 'Month', 'Week', 'Weekday', 'Day', 'DayOfYear']
    X = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    
    print("Treinando modelo CatBoost...")
    model = CatBoostRegressor(random_state=42, iterations=100, depth=5, silent=True)
    model.fit(X, y)
    
    print("Calculando SHAP Values...")
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)
    
    print(f"Gerando gráficos para {serie_name}...")
    
    # ---------------------------------------------------------
    # 1. Summary Plot (Sem título, com tradução)
    # ---------------------------------------------------------
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X, show=False)
    
    ax = plt.gca()
    ax.set_xlabel("Valor SHAP (impacto na previsão)", fontsize=22)
    
    # Traduzindo o Colorbar (Baixo/Alto e Valor da Variável)
    fig = plt.gcf()
    if len(fig.axes) > 1:
        cb_ax = fig.axes[-1]
        cb_ax.set_ylabel("Valor da Variável", fontsize=22)
        cb_ax.set_yticklabels(["Baixo", "Alto"], fontsize=14)
    
    plt.tight_layout()
    plt.savefig(f'figuras/shap_summary_{serie_name.lower()}.png', dpi=300, bbox_inches='tight')
    plt.close()

    # ---------------------------------------------------------
    # 2. Bar Plot (Sem título, com tradução)
    # ---------------------------------------------------------
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X, plot_type="bar", show=False)
    
    ax = plt.gca()
    ax.set_xlabel("Média(|Valor SHAP|) (impacto médio)", fontsize=22)
    
    plt.tight_layout()
    plt.savefig(f'figuras/shap_bar_{serie_name.lower()}.png', dpi=300, bbox_inches='tight')
    plt.close()

print("\nConcluído! Todos os gráficos formatados foram salvos na pasta 'figuras/'.")
